In [1]:
import torch
import random
import sys
import os
import re
import numpy as np


In [2]:
#loading modified skf file
Skf_File = os.path.join("../corrupt_pp", "C-C.skf")

# number of dftb training sets to generate and constraints for the random sampling of the training sets
N_SAMPLES=10
LOWER = -0.6
UPPER = 0.0

In [3]:
# automatically modify the skf file with random values for Ep, Es and SPE
# remark : actually, over 20 parameters should be modified, but for the sake of simplicity, we will only modify 3 parameters in this example. 0ver 20 parameters can not be optimized with BO, as I know.
def modify_skf(skf_file, output_file, Ep, Es, SPE):
    with open(skf_file, "r") as file:
            lines = file.readlines()
    line = lines[1]
    parameters = list(re.finditer( r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?", line ))
    replacements= [f"{Ep:.10f}", f"{Es:.10f}", f"{SPE:.10f}"]

    for parameter, replacement in reversed(list(zip(parameters[1:4], replacements))): 
          start, end = parameter.span()
          line = line[:start] + replacement + line[end:]
    lines[1] = line
    print("New Random ", Ep, ", ", Es, ", ",SPE)    
    with open(output_file, "w") as f:
          f.writelines(lines)

In [4]:
# generate the training set by modifying the skf file and running dftb+ for each set of parameters
import dataset
from scipy.stats import qmc
sampler = qmc.LatinHypercube(d=3)
l_bounds = [LOWER, LOWER, LOWER]
u_bounds = [UPPER, UPPER, UPPER]

input = []
output = []
for i in range(N_SAMPLES):
    sample = sampler.random(n=1)
    qmc.discrepancy(sample)
    sample = qmc.scale(sample, l_bounds, u_bounds)
    Ep = sample[0, 0]
    Es = sample[0, 1]
    SPE = sample[0, 2]
    modify_skf(Skf_File, Skf_File, Ep, Es, SPE) # new skf file is created with the random values for Ep, Es and SPE
    energy, force = dataset.test_md() # calculate the energy and forces using dftb for the modified skf file
    input_data = np.array([Ep, Es, SPE]) # create the input data array with the random values for Ep, Es and SPE   
    input.append(input_data) # append the input data to the input list for the training set
    output_data = np.array([energy, force]) # create the output data array with the calculated energy and forces
    output.append(output_data) # append the output data to the output list for the training set
input_tensor = torch.tensor(input, dtype=torch.float64) # create the input tensor for the training set
output_tensor = torch.tensor(output, dtype=torch.float64) # create the output tensor for the training set


    

New Random  -0.5721607791087365 ,  -0.5015484798118619 ,  -0.5476871340741359
Force RMSE: 1.962297 eV/Å
Energy RMSE: 97.051606 eV
New Random  -0.3427954161343158 ,  -0.07917359527872969 ,  -0.011303473505936745
Force RMSE: 0.639665 eV/Å
Energy RMSE: 11.047402 eV
New Random  -0.19414431346103 ,  -0.49037453826680877 ,  -0.14156902599711174
Force RMSE: 0.066747 eV/Å
Energy RMSE: 1.058248 eV
New Random  -0.10905581114674623 ,  -0.43333066649710383 ,  -0.3012253199591634
Force RMSE: 0.675223 eV/Å
Energy RMSE: 17.193273 eV
New Random  -0.1046154721077745 ,  -0.1864809087944122 ,  -0.5110081558894551
Force RMSE: 1.129624 eV/Å
Energy RMSE: 29.284835 eV
New Random  -0.4494312127612715 ,  -0.0590972325627539 ,  -0.5941086285626481
Force RMSE: 0.611160 eV/Å
Energy RMSE: 39.662597 eV
New Random  -0.39996678968927235 ,  -0.05161570927519299 ,  -0.23505348071589527
Force RMSE: 0.538888 eV/Å
Energy RMSE: 25.058620 eV
New Random  -0.5810668607486025 ,  -0.2490249355264051 ,  -0.3488933723459067
Force

/tmp/ipykernel_153856/4153083363.py:23: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:249.)
  input_tensor = torch.tensor(input, dtype=torch.float64) # create the input tensor for the training set


In [5]:
print("Input Tensor: ", input_tensor)
print("Output Tensor: ", output_tensor)


Input Tensor:  tensor([[-0.5722, -0.5015, -0.5477],
        [-0.3428, -0.0792, -0.0113],
        [-0.1941, -0.4904, -0.1416],
        [-0.1091, -0.4333, -0.3012],
        [-0.1046, -0.1865, -0.5110],
        [-0.4494, -0.0591, -0.5941],
        [-0.4000, -0.0516, -0.2351],
        [-0.5811, -0.2490, -0.3489],
        [-0.1277, -0.2970, -0.0585],
        [-0.0156, -0.3186, -0.3045]], dtype=torch.float64)
Output Tensor:  tensor([[9.7052e+01, 1.9623e+00],
        [1.1047e+01, 6.3967e-01],
        [1.0582e+00, 6.6747e-02],
        [1.7193e+01, 6.7522e-01],
        [2.9285e+01, 1.1296e+00],
        [3.9663e+01, 6.1116e-01],
        [2.5059e+01, 5.3889e-01],
        [9.1642e+01, 1.5782e+00],
        [2.1729e+01, 8.8328e-01],
        [3.5082e+01, 1.5524e+00]], dtype=torch.float64)


In [6]:
# BO
bounds = torch.tensor([[LOWER, UPPER], [LOWER, UPPER], [LOWER, UPPER]], dtype=torch.float64) # arbitrary bounds for the input parameters Ep, Es and SPE
#normalization and denormalization functions for the input parameters
def normalize(x):
    return (x - bounds[:, 0]) / (bounds[:, 1] - bounds[:, 0]) 
def denormalize(x):
    return x * (bounds[:, 1] - bounds[:, 0]) + bounds[:, 0]

train_x = normalize(input_tensor)


In [7]:
scale = torch.tensor([0.5, 0.5], dtype=torch.float64) # scale for the output parameters 1:1 Energy and Force

def calculate_loss(outputs):
    # Calculate the loss
    loss = torch.sum((outputs*scale), dim=1, keepdim=True)
    return loss

L = calculate_loss(output_tensor)

train_obj = -L

In [8]:
import os
import torch
import warnings
from typing import Optional

from botorch import fit_gpytorch_mll
from botorch.optim import optimize_acqf

from botorch.models import SingleTaskGP, ModelListGP
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood
from botorch.acquisition.objective import GenericMCObjective

from botorch.exceptions import BadInitialCandidatesWarning
from botorch.models.transforms.input import Normalize

from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.acquisition import qLogExpectedImprovement, qLogNoisyExpectedImprovement

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
SMOKE_TEST = os.environ.get("SMOKE_TEST")

# build model
def generate_model(train_x, train_obj, state_dict = None):
    model_obj = SingleTaskGP(
        train_x,
        train_obj,
        input_transform=Normalize(d=train_x.shape[-1]),
        ).to(train_x)

    model = ModelListGP(model_obj)
    mll = SumMarginalLogLikelihood(model.likelihood, model)
    if state_dict is not None:
        model.load_state_dict(state_dict)
    print("model is generated")
    return mll, model

mll, model = generate_model(train_x, train_obj)


/home/basic-user/src/pyc/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model is generated


In [9]:
def opimization_observation(acq_func, model, train_x, train_obj, num_restarts=10, raw_samples=512):
    # Optimize acquisition function
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=BadInitialCandidatesWarning)
        new_x, acq_value = optimize_acqf(
            acq_function=acq_func,
            bounds=torch.stack([torch.zeros(train_x.shape[-1], dtype=dtype), torch.ones(train_x.shape[-1], dtype=dtype)]),
            q=1,
            num_restarts=num_restarts,
            raw_samples=raw_samples,
        )
    new_x = new_x.detach()
    new_x_denormalized = denormalize(new_x)
    print("New x: ", new_x_denormalized)
    Ep, Es, SPE = new_x_denormalized[0].tolist()
    modify_skf(Skf_File, Skf_File, Ep, Es, SPE)
    energy, force = dataset.test_md()
    input_data = np.array([Ep, Es, SPE])    
    output_data = np.array([energy, force])
    input_tensor_new = torch.tensor(input_data.reshape(1,-1), dtype=torch.float64)
    output_tensor_new = torch.tensor(output_data.reshape(1,-1), dtype=torch.float64)
    train_x_new = torch.cat((train_x, normalize(input_tensor_new)), dim=0)
    train_obj_new = torch.cat((train_obj, -calculate_loss(output_tensor_new)), dim=0)
    print("New train_x: ", input_tensor_new)
    print("New train_obj: ", calculate_loss(output_tensor_new))
    return train_x_new, train_obj_new

new_x, new_obj = opimization_observation(qLogExpectedImprovement(model=model, best_f=train_obj.max()), model, train_x, train_obj)
mll_new, model_new = generate_model(new_x, new_obj)

New x:  tensor([[-0.0876, -0.5863, -0.1367]], dtype=torch.float64)
New Random  -0.0875670683275862 ,  -0.5863351185041268 ,  -0.13673113985312724
Force RMSE: 0.528576 eV/Å
Energy RMSE: 8.185615 eV
New train_x:  tensor([[-0.0876, -0.5863, -0.1367]], dtype=torch.float64)
New train_obj:  tensor([[4.3571]], dtype=torch.float64)
model is generated


In [10]:
print(new_x, new_obj)

tensor([[0.0464, 0.1641, 0.0872],
        [0.4287, 0.8680, 0.9812],
        [0.6764, 0.1827, 0.7641],
        [0.8182, 0.2778, 0.4980],
        [0.8256, 0.6892, 0.1483],
        [0.2509, 0.9015, 0.0098],
        [0.3334, 0.9140, 0.6082],
        [0.0316, 0.5850, 0.4185],
        [0.7872, 0.5051, 0.9024],
        [0.9740, 0.4690, 0.4925],
        [0.8541, 0.0228, 0.7721]], dtype=torch.float64) tensor([[-49.5070],
        [ -5.8435],
        [ -0.5625],
        [ -8.9342],
        [-15.2072],
        [-20.1369],
        [-12.7988],
        [-46.6103],
        [-11.3063],
        [-18.3172],
        [ -4.3571]], dtype=torch.float64)


In [11]:
while np.abs(new_obj.max()) > 0.001:
    new_x, new_obj = opimization_observation(qLogExpectedImprovement(model=model_new, best_f=train_obj.max()), model, new_x, new_obj)
    mll_new, model_new = generate_model(new_x, new_obj)

print("x = ", denormalize(new_x), "RSME = ", new_obj)

New x:  tensor([[-0.2831, -0.5834,  0.0000]], dtype=torch.float64)
New Random  -0.2831368964461439 ,  -0.583437149301715 ,  0.0


/tmp/ipykernel_153856/3237689895.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


Force RMSE: 0.619055 eV/Å
Energy RMSE: 21.772854 eV
New train_x:  tensor([[-0.2831, -0.5834,  0.0000]], dtype=torch.float64)
New train_obj:  tensor([[11.1960]], dtype=torch.float64)
model is generated
New x:  tensor([[-0.1861,  0.0000, -0.1298]], dtype=torch.float64)
New Random  -0.18613844811508112 ,  0.0 ,  -0.12979332685701694


/tmp/ipykernel_153856/3237689895.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


Force RMSE: 1.014102 eV/Å
Energy RMSE: 22.065627 eV
New train_x:  tensor([[-0.1861,  0.0000, -0.1298]], dtype=torch.float64)
New train_obj:  tensor([[11.5399]], dtype=torch.float64)
model is generated
New x:  tensor([[-0.2194, -0.6000, -0.2325]], dtype=torch.float64)
New Random  -0.21940066096061966 ,  -0.6 ,  -0.23245158069014998


/tmp/ipykernel_153856/3237689895.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


... (2,897 outputs left out)

/tmp/ipykernel_153856/3237689895.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


New x:  tensor([[-0.2324, -0.3974, -0.2254]], dtype=torch.float64)
New Random  -0.23236771699474668 ,  -0.39743507084699536 ,  -0.2253524130128
Force RMSE: 0.339759 eV/Å
Energy RMSE: 0.282749 eV
New train_x:  tensor([[-0.2324, -0.3974, -0.2254]], dtype=torch.float64)
New train_obj:  tensor([[0.3113]], dtype=torch.float64)
model is generated


/tmp/ipykernel_153856/3237689895.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


KeyboardInterrupt: 

In [14]:
f, e = dataset.test_md('../trans3d-0-1/')
print("Input: ", input_data)
print("Output: ", 0.5*f+0.5*e)

Force RMSE: 0.000000 eV/Å
Energy RMSE: 0.000000 eV
Input:  [-0.01560073 -0.31862159 -0.30448505]
Output:  2.0691260746716837e-16
